In [53]:
from dotenv import load_dotenv
import os

load_dotenv()

google_api_key = os.getenv("GEMINI_API_KEY")

In [54]:

from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
# this is a human message
response = model.invoke("Hello, how are you?")
# this will generate a ai message
response

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    genre: str = Field(..., description="The genre of the movie")
    director: str = Field(..., description="The director of the movie")

In [35]:
model_with_structure = model.with_structured_output(Movie)

In [36]:
response = model_with_structure.invoke("Tell me about the movie Inception.")
response 

Movie(title='Inception', year=2010, genre='Science Fiction', director='Christopher Nolan')

In [37]:
# NESTED STRUCTURE

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title: str
    year: int
    genre: str
    director: str
    actors: list[Actor]



In [38]:
model_with_structure = model.with_structured_output(MovieDetails)

In [39]:
response = model_with_structure.invoke("details about movie Inception.")

In [40]:
response 

MovieDetails(title='Inception', year=2010, genre='Science Fiction', director='Christopher Nolan', actors=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Ken Watanabe', role='Saito')])

### TYPEDICT
typeddict provides a simpler alternative using pyhton built in typing ideal when you dont need runtime validation

In [41]:
from typing_extensions import TypedDict,Annotated


In [42]:
class MovieDict(TypedDict):
    title: str
    year: int
    genre: str
    director: str

model_withtypeddict = model.with_structured_output(MovieDict)

In [43]:
response = model_withtypeddict.invoke("details about movie Inception.")
response 


{'title': 'Inception',
 'year': 2010,
 'genre': 'Science Fiction',
 'director': 'Christopher Nolan'}

In [44]:
model.profile 

{'name': 'Gemini 2.5 Flash',
 'release_date': '2025-03-20',
 'last_updated': '2025-06-05',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True}

In [48]:
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str

agent = create_agent(model, response_format=ContactInfo)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Extract contact information from: JOhn Doe, john@exaple.com, 123-456-7890."}
    ]
})
print(result["structured_response"])

name='JOhn Doe' email='john@exaple.com' phone='123-456-7890'


In [50]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str

agent = create_agent(model, response_format=ContactInfo)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Extract contact information from: JOhn Doe, john@exaple.com, 123-456-7890."}
    ]
})
print(result["structured_response"])


ContactInfo(name='JOhn Doe', email='john@exaple.com', phone='123-456-7890')
